In [ ]:
# Parameters cell in Jupyter Notebook
stats_category = "default_category"  # This default value will be replaced by Papermill. Possible values: 'allPlayers', 'player', 'allClubs', 'club', 'competition', 'game'
identifier = None  # Default value, can be empty if not required

In [ ]:
# Load environment variables
import dotenv

dotenv.load_dotenv(".env")

# Extract connection info
import os

mongodb_uri = os.getenv("MONGODB_URI")
postgres_host = os.getenv("POSTGRES_HOST")
postgres_port = os.getenv("POSTGRES_PORT")
postgres_db = os.getenv("POSTGRES_DB")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")

# Connect to MongoDB
from pymongo import MongoClient

mongo_client = MongoClient(mongodb_uri)
db = mongo_client["fdpdata"]

# Connect to PostgreSQL
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql://{postgres_user}:{postgres_password}@{postgres_host}:{postgres_port}/{postgres_db}"
)


# Import commonly used libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


# Check category first
if stats_category != "competition" and stats_category != "default_category":
    # Check if identifier is not None and cast it as int
    if identifier is not None:
        identifier = int(identifier)
elif stats_category == "competition":
    # Check if identifier is not None and cast it as str
    if identifier is not None:
        identifier = str(identifier)

In [ ]:
from xml.etree import ElementTree as ET


def tag_as_advanced_statistic(svg_file_path):
    # Load the SVG file
    tree = ET.parse(svg_file_path)
    root = tree.getroot()

    # Add the custom attribute
    root.set("data-statistic-type", "advanced")

    # Save the modified SVG
    tree.write(svg_file_path)

# Players


## All Players


In [ ]:
# Cell: All Players Analysis
if stats_category == "allPlayers":
    # Ensure the output directory exists
    output_dir = "./outputs/allPlayers"
    os.makedirs(output_dir, exist_ok=True)

    # Load data
    players_data = pd.read_sql("SELECT * from players", engine)
    appearances_collection = db.appearances

    # Aggregating total goals by player
    appearances_pipeline = [
        {"$group": {"_id": "$player_id", "total_goals": {"$sum": "$goals"}}}
    ]
    appearances_data = list(appearances_collection.aggregate(appearances_pipeline))
    appearances_df = pd.DataFrame(appearances_data)
    appearances_df.rename(columns={"_id": "player_id"}, inplace=True)

    # Merging data
    players_data = players_data.merge(appearances_df, on="player_id", how="left")

    # Age Distribution Analysis
    plt.figure(figsize=(10, 6))
    sns.histplot(players_data["age"], bins=20, kde=True)
    plt.title("Age Distribution of Players")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.savefig(os.path.join(output_dir, "age_distribution.svg"))

    # Positional Analysis
    positions_count = players_data["position"].value_counts()
    plt.figure(figsize=(10, 6))
    positions_count.plot(kind="bar")
    plt.title("Player Count by Position")
    plt.xlabel("Position")
    plt.ylabel("Count")
    filename = "player_count_by_position.svg"
    plt.savefig(os.path.join(output_dir, filename))

    # Club Affiliation Diversity - Top 10
    plt.figure(figsize=(10, 6))
    top_clubs = players_data["current_club_name"].value_counts().head(10)
    top_clubs.plot(kind="bar")
    plt.title("Top 10 Clubs by Historical Player Count")
    plt.xlabel("Club")
    plt.ylabel("Player Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    filename = "top_clubs_by_historical_player_count.svg"
    plt.savefig(os.path.join(output_dir, filename))

    # Advanced Statistics

    # Market Value Analysis
    plt.figure(figsize=(12, 8))
    sns.boxplot(
        data=players_data,
        x="position",
        y="market_value_in_eur",
        whis=[5, 95],
        showfliers=False,
    )
    plt.title("Market Value by Position", fontsize=16)
    plt.xlabel("Position", fontsize=14)
    plt.ylabel("Market Value (EUR)", fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    filename = "market_value_by_position.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

    # Physical Attributes Correlation
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=players_data, x="height_in_cm", y="market_value_in_eur")
    plt.title("Height vs Market Value")
    plt.xlabel("Height (cm)")
    plt.ylabel("Market Value (EUR)")
    filename = "height_vs_market_value.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

    # Performance and Age Relationship
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=players_data, x="age", y="total_goals")
    plt.title("Performance vs Age")
    plt.xlabel("Age")
    plt.ylabel("Total Goals")
    filename = "performance_vs_age.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))
    
    import geopandas as gpd
    
    # Load world map
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    
    players_data['country_of_citizenship'] = players_data['country_of_citizenship'].replace({
    'USA': 'United States of America',
    'United States': 'United States of America',
    'England': 'United Kingdom',
    'Scotland': 'United Kingdom',
    'Wales': 'United Kingdom',
    'Northern Ireland': 'United Kingdom',
    'Korea, South': 'South Korea',
    'Korea, North': 'North Korea',
    'DR Congo': 'Congo'
    })

    # Aggregate player count by country
    country_counts = players_data['country_of_citizenship'].value_counts().reset_index()
    country_counts.columns = ['country', 'player_count']

    # Merge this with the world map data
    world = world.merge(country_counts, how='left', left_on='name', right_on='country')

    # Plotting
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    base = world.plot(ax=ax, color='white', edgecolor='black')
    world.dropna(subset=['player_count']).plot(column='player_count', ax=base, legend=True, cmap='coolwarm',
                                            legend_kwds={'label': "Player Count by Country",
                                                            'orientation': "horizontal"})


    plt.title('Global Distribution of Players by Nationality')
    plt.savefig(os.path.join(output_dir, "player_distribution_by_nationality.svg"))    
    
    players_data['market_value_in_eur'] = pd.to_numeric(players_data['market_value_in_eur'], errors='coerce')
    market_value_by_country = players_data.groupby('country_of_citizenship')['market_value_in_eur'].sum().reset_index()
    market_value_by_country.columns = ['country', 'total_market_value']
    
    world_with_values = world.merge(market_value_by_country, how='left', left_on='name', right_on='country')
    
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    base = world_with_values.plot(ax=ax, color='lightgrey', edgecolor='black')
    world_with_values.dropna(subset=['total_market_value']).plot(column='total_market_value', ax=base, legend=True, cmap='coolwarm',
                                                                legend_kwds={'label': "Total Market Value by Country",
                                                                            'orientation': "horizontal"})
    plt.title('Global Distribution of Players\' Market Value')
    plt.savefig(os.path.join(output_dir, "players_market_value_distribution.svg"))
    tag_as_advanced_statistic(os.path.join(output_dir, "players_market_value_distribution.svg"))
    
    # Cards Analysis: Infractions by country
    # Fetching all game events data of type "Cards"
    cards_pipeline = [
        {"$match": {"type": "Cards"}},
        {"$group": {"_id": "$player_id", "total_infractions": {"$sum": 1}}}
    ]
    cards_data = list(db.game_events.aggregate(cards_pipeline))
    cards_df = pd.DataFrame(cards_data)

    # Rename '_id' to 'player_id' for consistency
    cards_df.rename(columns={"_id": "player_id"}, inplace=True)

    # Ensure 'player_id' column exists in players_data, or adapt the data as needed
    if 'player_id' not in players_data.columns:
        print("player_id column is missing in players_data dataframe. Please check your data.")

    # Merging the cards data with player data to get country of citizenship
    players_with_cards = players_data.merge(cards_df, on="player_id", how="left")

    # Handling NaN values for players with no infractions
    players_with_cards['total_infractions'] = players_with_cards['total_infractions'].fillna(0)

    # Aggregating total infractions by country
    country_infractions = players_with_cards.groupby('country_of_citizenship').agg({'total_infractions': 'sum'}).reset_index()

    # Merging this data with the world map
    world_with_infractions = world.merge(country_infractions, how='left', left_on='name', right_on='country_of_citizenship')

    # Plotting
    fig, ax = plt.subplots(1, figsize=(20, 12))
    world_with_infractions.boundary.plot(ax=ax, linewidth=1)
    world_with_infractions.dropna(subset=['total_infractions']).plot(column='total_infractions', ax=ax, legend=True, cmap='Reds',
                                                        legend_kwds={'label': "Total Infractions by Country",
                                                                    'orientation': "horizontal"})
    plt.title('Global Distribution of Football Infractions by Player Nationality')
    plt.savefig(os.path.join(output_dir, "infractions_distribution_by_nationality.svg"))
    tag_as_advanced_statistic(os.path.join(output_dir, "infractions_distribution_by_nationality.svg"))




## Single Player


In [ ]:
# Cell: Single Player Analysis
if stats_category == "player" and identifier:
    # Ensure the output directory exists
    output_dir = f"./outputs/player/{identifier}"
    os.makedirs(output_dir, exist_ok=True)

    # Load player data from PostgreSQL
    player_query = f"SELECT * FROM players WHERE player_id = {identifier}"
    player_data = pd.read_sql(player_query, engine)

    # Load valuation data from MongoDB
    valuations_pipeline = [
        {
            "$match": {
                "player_id": identifier
            }
        },
        {
            "$project": {
                "_id": 0,
                "player_id": 1,
                "market_value_in_eur": 1,
                "date": {
                    "$dateToString": {"format": "%Y-%m-%d", "date": "$date"}
                },  # Convert date to string
            }
        },
        {"$sort": {"date": 1}},  # Sort by date to get a chronological order
    ]

    valuations_collection = db.player_valuations
    valuations_data = list(valuations_collection.aggregate(valuations_pipeline))
    valuations_df = pd.DataFrame(valuations_data)

    # Merging data to include player info with valuations
    player_valuations = pd.merge(player_data, valuations_df, on="player_id", how="left")

    # Create the visualization
    plt.figure(figsize=(10, 6))
    plt.plot(
        player_valuations["date"],
        player_valuations["market_value_in_eur_y"],
        marker="o",
    )
    plt.title(f"Market Value Over Time for {player_data['name'][0]}")
    plt.xlabel("Date")
    plt.ylabel("Market Value (MM EUR)")
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"market_value_{identifier}.svg"))

    # Load appearances data from MongoDB
    appearances_pipeline = [
        {"$match": {"player_id": identifier}},
        {
            "$lookup": {
                "from": "games",
                "localField": "game_id",
                "foreignField": "game_id",
                "as": "game_info",
            }
        },
        {"$unwind": "$game_info"},
        {
            "$group": {
                "_id": "$game_info.season",
                "total_goals": {"$sum": "$goals"},
                "total_assists": {"$sum": "$assists"},
                "appearances": {"$sum": 1},
                "yellow_cards": {"$sum": "$yellow_cards"},
                "red_cards": {"$sum": "$red_cards"},
            }
        },
        {"$sort": {"_id": 1}},
    ]

    appearances_data = list(db.appearances.aggregate(appearances_pipeline))
    performance_df = pd.DataFrame(appearances_data)

    # Create the visualization for Performance over Seasons and Disciplinary Record
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Performance data
    ax1.set_title(
        f"Performance and Disciplinary Record over Seasons for {player_data['name'][0]}"
    )
    ax1.plot(
        performance_df["_id"],
        performance_df["total_goals"],
        label="Total Goals",
        marker="o",
        color="blue",
    )
    ax1.plot(
        performance_df["_id"],
        performance_df["total_assists"],
        label="Total Assists",
        marker="o",
        color="green",
    )
    ax1.set_xlabel("Season")
    ax1.set_ylabel("Performance Metrics", color="blue")
    ax1.tick_params(axis="y", labelcolor="blue")

    # Disciplinary data
    ax2 = ax1.twinx()
    ax2.plot(
        performance_df["_id"],
        performance_df["yellow_cards"],
        label="Yellow Cards",
        marker="o",
        color="yellow",
    )
    ax2.plot(
        performance_df["_id"],
        performance_df["red_cards"],
        label="Red Cards",
        marker="o",
        color="red",
    )
    ax2.set_ylabel("Card Count", color="red")
    ax2.tick_params(axis="y", labelcolor="red")

    # Legends and layout
    fig.tight_layout()
    fig.legend(loc="upper left", bbox_to_anchor=(0, 1), bbox_transform=ax1.transAxes)

    # Save and show the visualization
    plt.savefig(os.path.join(output_dir, f"performance_disciplinary_{identifier}.svg"))

    print(f"Performed analysis for player {identifier}")

    # Advanced Statistics

    # Comparison with Averages
    # We will compare the player's total goals and assists with the league average.

    league_averages_pipeline = [
        {
            "$group": {
                "_id": "$season",
                "avg_goals": {"$avg": "$goals"},
                "avg_assists": {"$avg": "$assists"},
            }
        },
        {"$sort": {"_id": 1}},
    ]

    league_averages_data = list(db.appearances.aggregate(league_averages_pipeline))
    league_averages_df = pd.DataFrame(league_averages_data)

    # Visualization for Comparison with Averages
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.set_title(f"Player vs League Averages for {player_data['name'][0]}")
    ax1.plot(
        performance_df["_id"],
        performance_df["total_goals"],
        label="Player Total Goals",
        marker="o",
        color="blue",
    )
    ax1.plot(
        league_averages_df["_id"],
        league_averages_df["avg_goals"],
        label="League Avg Goals",
        marker="^",
        color="green",
        linestyle="--",
    )
    ax1.set_xlabel("Season")
    ax1.set_ylabel("Goals", color="blue")
    ax1.tick_params(axis="y", labelcolor="blue")

    ax2 = ax1.twinx()
    ax2.plot(
        performance_df["_id"],
        performance_df["total_assists"],
        label="Player Total Assists",
        marker="o",
        color="red",
    )
    ax2.plot(
        league_averages_df["_id"],
        league_averages_df["avg_assists"],
        label="League Avg Assists",
        marker="^",
        color="purple",
        linestyle="--",
    )
    ax2.set_ylabel("Assists", color="red")
    ax2.tick_params(axis="y", labelcolor="red")

    fig.tight_layout()
    fig.legend(loc="upper left", bbox_to_anchor=(0, 1), bbox_transform=ax1.transAxes)
    filename = f"player_vs_league_averages_{identifier}.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

    print(f"Advanced analysis completed for player {identifier}")

# Clubs


## All Clubs


In [ ]:
# Cell: All Clubs Analysis
if stats_category == "allClubs":
    # Ensure the output directory exists
    output_dir = "./outputs/allClubs"
    os.makedirs(output_dir, exist_ok=True)

    # Club Market Value Distribution
    market_value_query = """
    SELECT current_club_id, AVG(market_value_in_eur) as avg_market_value
    FROM players
    GROUP BY current_club_id
    """
    market_values = pd.read_sql(market_value_query, engine)

    # Plotting the market value distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(market_values["avg_market_value"], bins=20, kde=True)
    plt.title("Average Market Value Distribution Across Clubs")
    plt.xlabel("Average Market Value (MM EUR)")
    plt.ylabel("Number of Clubs")
    plt.savefig(os.path.join(output_dir, "market_value_distribution.svg"))

    # Club Age Profile
    age_profile_query = """
    SELECT current_club_name, AVG(age) as avg_age
    FROM players
    GROUP BY current_club_name
    ORDER BY avg_age DESC
    LIMIT 15
    """
    age_profiles = pd.read_sql(age_profile_query, engine)

    # Plotting the age profile
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x=age_profiles["current_club_name"],
        y=age_profiles["avg_age"],
        hue=age_profiles["current_club_name"],
        palette="viridis",
        legend=False,
    )
    plt.title("Average Age Profile Across Clubs")
    plt.xlabel("Club Name")
    plt.ylabel("Average Age")
    plt.xticks(rotation=90)
    plt.savefig(os.path.join(output_dir, "age_profile.svg"))

    # Advanced Statistics
    clubs_query = "SELECT club_id, name FROM clubs"
    clubs_data = pd.read_sql(clubs_query, engine)

    games_pipeline = [
        {
            "$group": {
                "_id": {
                    "club_id": {
                        "$cond": [
                            {"$eq": ["$side", "home"]},
                            "$home_club_id",
                            "$away_club_id",
                        ]
                    },
                    "season": "$season",
                },
                "goals": {
                    "$sum": {
                        "$cond": [
                            {"$eq": ["$side", "home"]},
                            "$home_club_goals",
                            "$away_club_goals",
                        ]
                    }
                },
            }
        },
        {
            "$project": {
                "_id": 0,
                "club_id": "$_id.club_id",
                "season": "$_id.season",
                "total_goals": "$goals",
            }
        },
        {"$sort": {"season": 1, "club_id": 1}},  # Sorting for easier verification
    ]

    games_data = list(db.games.aggregate(games_pipeline))
    games_df = pd.DataFrame(games_data)

    # Merge with club names
    club_performance = pd.merge(games_df, clubs_data, on="club_id")
    club_performance = club_performance[
        club_performance["season"] <= 2022
    ]  # Filter out the current season due to lack of data

    # Pivot data for visualization
    club_performance_pivot = club_performance.pivot_table(
        index="season", columns="name", values="total_goals", fill_value=0
    )

    #  Aggregate the total goals per club across all seasons.
    total_goals_by_club = (
        club_performance.groupby("club_id")["total_goals"].sum().reset_index()
    )

    # Get the top 10 clubs by total goals scored.
    top_clubs = total_goals_by_club.sort_values(by="total_goals", ascending=False).head(
        10
    )

    # Now we filter the club_performance DataFrame to only include these top clubs.
    top_club_ids = top_clubs["club_id"].tolist()
    top_clubs_performance = club_performance[
        club_performance["club_id"].isin(top_club_ids)
    ]

    # We need to map the club names to their ids to order the legend.
    club_name_map = clubs_data.set_index("club_id")["name"].to_dict()

    # Create a pivot table for the top clubs.
    # We ensure the columns (club names) are ordered by total goals by mapping the club ids back to names.
    top_clubs_performance_pivot = top_clubs_performance.pivot_table(
        index="season", columns="club_id", values="total_goals", fill_value=0
    )[
        top_club_ids
    ]  # This reorders the columns based on the total goals scored

    # Replace club_id with club names for the columns
    top_clubs_performance_pivot.columns = [
        club_name_map[club_id] for club_id in top_clubs_performance_pivot.columns
    ]

    # Plotting the graph.
    plt.figure(figsize=(20, 10))
    sns.lineplot(
        data=top_clubs_performance_pivot, dashes=False, palette="bright", linewidth=2.5
    )

    # Sorting the legend based on total goals and creating custom handles.
    handles, labels = plt.gca().get_legend_handles_labels()
    labels_handles = dict(zip(labels, handles))
    sorted_labels_handles = dict(
        sorted(
            labels_handles.items(),
            key=lambda lh: top_club_ids.index(
                int(clubs_data[clubs_data["name"] == lh[0]]["club_id"].iloc[0])
            ),
        )
    )

    plt.title("Top 10 Clubs Performance Over Time - Total Goals per Season")
    plt.xlabel("Season")
    plt.ylabel("Total Goals Scored")
    plt.legend(
        sorted_labels_handles.values(),
        sorted_labels_handles.keys(),
        title="Club",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        labelspacing=1.86,
        prop={"size": 13},
    )
    plt.xticks(rotation=45)
    plt.tight_layout()
    filename = "top_clubs_performance.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

    games_pipeline = [
        {
            # Unwind the games into individual documents for home and away clubs
            "$facet": {
                "home_games": [
                    {
                        "$project": {
                            "club_id": "$home_club_id",
                            "goals_scored": "$home_club_goals",
                            "goals_conceded": "$away_club_goals",
                            "win": {"$cond": [{"$eq": ["$outcome", "Home Win"]}, 1, 0]},
                            "loss": {
                                "$cond": [{"$eq": ["$outcome", "Away Win"]}, 1, 0]
                            },
                            "draw": {"$cond": [{"$eq": ["$outcome", "Draw"]}, 1, 0]},
                            "total_games": {"$literal": 1},
                        }
                    }
                ],
                "away_games": [
                    {
                        "$project": {
                            "club_id": "$away_club_id",
                            "goals_scored": "$away_club_goals",
                            "goals_conceded": "$home_club_goals",
                            "win": {"$cond": [{"$eq": ["$outcome", "Away Win"]}, 1, 0]},
                            "loss": {
                                "$cond": [{"$eq": ["$outcome", "Home Win"]}, 1, 0]
                            },
                            "draw": {"$cond": [{"$eq": ["$outcome", "Draw"]}, 1, 0]},
                            "total_games": {"$literal": 1},
                        }
                    }
                ],
            }
        },
        {
            # Combine home and away games into one array
            "$project": {"all_games": {"$concatArrays": ["$home_games", "$away_games"]}}
        },
        {
            # Flatten the combined array
            "$unwind": "$all_games"
        },
        {
            # Replace the document root with the game documents
            "$replaceRoot": {"newRoot": "$all_games"}
        },
        {
            # Group by club_id and calculate the required metrics
            "$group": {
                "_id": "$club_id",
                "total_games": {"$sum": "$total_games"},
                "total_wins": {"$sum": "$win"},
                "total_losses": {"$sum": "$loss"},
                "total_draws": {"$sum": "$draw"},
                "avg_goals_scored": {"$avg": "$goals_scored"},
                "avg_goals_conceded": {"$avg": "$goals_conceded"},
            }
        },
        {
            # Project the final structure
            "$project": {
                "_id": 0,
                "club_id": "$_id",
                "win_loss_ratio": {
                    "$cond": [
                        {"$ne": ["$total_games", 0]},
                        {"$divide": ["$total_wins", "$total_games"]},
                        0,
                    ]
                },
                "avg_goals_scored": 1,
                "avg_goals_conceded": 1,
            }
        },
    ]

    # Execute the aggregation pipeline
    club_stats = list(db.games.aggregate(games_pipeline))
    club_stats_df = pd.DataFrame(club_stats)

    # Load club names from PostgreSQL
    clubs_query = "SELECT club_id, name FROM clubs"
    clubs_data = pd.read_sql(clubs_query, engine)

    # Merge the club stats with the club names
    club_stats_with_names = pd.merge(club_stats_df, clubs_data, on="club_id")

    # Sort clubs by their win-loss ratio in descending order and select the top N
    top_club_stats_df = club_stats_df.sort_values(
        by="win_loss_ratio", ascending=False
    ).head(10)

    # Merge the top club stats with the club names
    top_club_stats_with_names = pd.merge(top_club_stats_df, clubs_data, on="club_id")

    # Plotting Win-Loss Ratio vs Average Goals Scored and Conceded for top clubs
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        x="avg_goals_scored",
        y="avg_goals_conceded",
        size="win_loss_ratio",
        hue="name",  # Color points by club name for identification
        data=top_club_stats_with_names,
        sizes=(100, 500),
        alpha=0.7,
        legend="auto",
        palette="bright"
    )

    plt.title("Top Clubs by Win-Loss Ratio: Average Goals Scored vs Conceded")
    plt.xlabel("Average Goals Scored")
    plt.ylabel("Average Goals Conceded")
    plt.legend(
        bbox_to_anchor=(1.05, 1), loc="upper left", labelspacing=1.86, prop={"size": 8}
    )
    plt.tight_layout()
    filename = "top_clubs_win_loss_ratio.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

## Single Club


In [ ]:
# Cell: Single Club Analysis
if stats_category == "club" and identifier:
    # Ensure the output directory exists
    output_dir = f"./outputs/club/{identifier}"
    os.makedirs(output_dir, exist_ok=True)

    performance_pipeline = [
        {
            "$match": {
                "$or": [{"home_club_id": identifier}, {"away_club_id": identifier}]
            }
        },
        {
            "$project": {
                "season": 1,
                "win": {
                    "$cond": [
                        {
                            "$or": [
                                {
                                    "$and": [
                                        {"$eq": ["$home_club_id", identifier]},
                                        {"$eq": ["$outcome", "Home Win"]},
                                    ]
                                },
                                {
                                    "$and": [
                                        {"$eq": ["$away_club_id", identifier]},
                                        {"$eq": ["$outcome", "Away Win"]},
                                    ]
                                },
                            ]
                        },
                        1,
                        0,
                    ]
                },
                "draw": {"$cond": [{"$eq": ["$outcome", "Draw"]}, 1, 0]},
                "loss": {
                    "$cond": [
                        {
                            "$or": [
                                {
                                    "$and": [
                                        {"$eq": ["$home_club_id", identifier]},
                                        {"$eq": ["$outcome", "Away Win"]},
                                    ]
                                },
                                {
                                    "$and": [
                                        {"$eq": ["$away_club_id", identifier]},
                                        {"$eq": ["$outcome", "Home Win"]},
                                    ]
                                },
                            ]
                        },
                        1,
                        0,
                    ]
                },
            }
        },
        {
            "$group": {
                "_id": "$season",
                "wins": {"$sum": "$win"},
                "draws": {"$sum": "$draw"},
                "losses": {"$sum": "$loss"},
            }
        },
        {"$sort": {"_id": 1}},
    ]

    # Execute the aggregation pipeline
    season_performance_data = list(db.games.aggregate(performance_pipeline))
    season_performance_df = pd.DataFrame(season_performance_data)
    # Rename '_id' to 'season' to reflect the content accurately
    season_performance_df.rename(columns={"_id": "season"}, inplace=True)

    plt.figure(figsize=(10, 6))
    plt.plot(
        season_performance_df["season"],
        season_performance_df["wins"],
        label="Wins",
        marker="o",
    )
    plt.plot(
        season_performance_df["season"],
        season_performance_df["draws"],
        label="Draws",
        marker="s",
    )
    plt.plot(
        season_performance_df["season"],
        season_performance_df["losses"],
        label="Losses",
        marker="x",
    )
    plt.title("Season Performance Over Time")
    plt.xlabel("Season")
    plt.ylabel("Number of Matches")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f"season_performance_{identifier}.svg"))

    # MongoDB pipeline for Top Scorers
    top_scorers_pipeline = [
        {"$match": {"player_current_club_id": identifier}},
        {
            "$group": {
                "_id": "$player_id",
                "total_goals": {"$sum": "$goals"},
                "player_name": {"$first": "$player_name"},
            }
        },
        {"$sort": {"total_goals": -1}},
        {
            "$limit": 10
        },
    ]

    # Execute the pipeline
    top_scorers_data = list(db.appearances.aggregate(top_scorers_pipeline))
    top_scorers_df = pd.DataFrame(top_scorers_data)

    # Visualization for Top Scorers
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x="total_goals",
        y="player_name",
        hue="player_name",
        data=top_scorers_df,
        palette="viridis",
        legend=False,
    )
    plt.title("Top Scorers")
    plt.xlabel("Total Goals")
    plt.ylabel("Player Name")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"top_scorers_{identifier}.svg"))

    # SQL query for Age Distribution
    age_distribution_query = (
        f"SELECT age FROM players WHERE current_club_id = {identifier}"
    )
    age_distribution_df = pd.read_sql(age_distribution_query, engine)

    # Visualization for Age Distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(age_distribution_df["age"], bins=15, kde=True, color="skyblue")
    plt.title(f"Age Distribution of Players")
    plt.xlabel("Age")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"age_distribution_{identifier}.svg"))

    # Advanced Statistics
    # MongoDB pipeline for Goal Difference Per Match
    goal_diff_pipeline = [
        {
            "$match": {
                "$or": [{"home_club_id": identifier}, {"away_club_id": identifier}]
            }
        },
        {
            "$project": {
                "season": 1,
                "date": 1,
                "goal_difference": {
                    "$cond": [
                        {"$eq": ["$home_club_id", identifier]},
                        {"$subtract": ["$home_club_goals", "$away_club_goals"]},
                        {"$subtract": ["$away_club_goals", "$home_club_goals"]},
                    ]
                },
            }
        },
        {
            "$sort": {"season": 1, "date": 1}
        },  # Sort by season and date for chronological order
    ]

    # Execute the pipeline
    goal_diff_data = list(db.games.aggregate(goal_diff_pipeline))
    goal_diff_df = pd.DataFrame(goal_diff_data)

    # Add a sequential match index for plotting
    goal_diff_df["match_index"] = range(1, len(goal_diff_df) + 1)

    # Visualization for Goal Difference Per Match
    plt.figure(figsize=(15, 6))
    sns.lineplot(
        x="match_index",
        y="goal_difference",
        hue="season",
        data=goal_diff_df,
        palette="tab10",
        marker="o",
    )
    plt.title("Goal Difference Per Match for Club")
    plt.xlabel("Match Index")
    plt.ylabel("Goal Difference")
    plt.legend(title="Season")
    plt.tight_layout()
    filename = f"goal_difference_{identifier}.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

    # MongoDB pipeline for Player Contribution to Wins
    player_contribution_pipeline = [
        # Start with club_games to reduce initial dataset size
        {
            "$match": {"is_win": 1, "club_id": identifier}
        },  # Filter for winning games of the club
        {
            "$lookup": {  # Join with appearances to get player performance
                "from": "appearances",
                "localField": "game_id",
                "foreignField": "game_id",
                "as": "player_performance",
            }
        },
        {"$unwind": "$player_performance"},  # Unwind the joined documents
        {
            "$match": {"player_performance.player_current_club_id": identifier}
        },  # Filter for club players
        {
            "$group": {
                "_id": "$player_performance.player_id",
                "total_goals": {"$sum": "$player_performance.goals"},
                "total_assists": {"$sum": "$player_performance.assists"},
                "player_name": {"$first": "$player_performance.player_name"},
            }
        },
        {
            "$sort": {"total_goals": -1, "total_assists": -1}
        },  # Sort by goals and assists
        {"$limit": 10},  # Limit to top contributors
    ]

    # Execute the pipeline
    player_contrib_data = list(db.club_games.aggregate(player_contribution_pipeline))
    player_contrib_df = pd.DataFrame(player_contrib_data)

    # Visualization for Player Contribution to Wins
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x="total_goals",
        y="player_name",
        data=player_contrib_df,
        color="blue",
        label="Goals",
    )
    sns.barplot(
        x="total_assists",
        y="player_name",
        data=player_contrib_df,
        color="orange",
        label="Assists",
    )
    plt.title("Player Contribution to Wins for Club")
    plt.xlabel("Total Contributions (Goals + Assists)")
    plt.ylabel("Player Name")
    plt.legend()
    plt.tight_layout()
    filename = f"player_contribution_{identifier}.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

# Competitions


## Single Competition


In [ ]:
# Cell: Single Competition Analysis
if stats_category == "competition" and identifier:
    # Ensure the output directory exists
    output_dir = f"./outputs/competition/{identifier}"
    os.makedirs(output_dir, exist_ok=True)

    goals_pipeline = [
        {"$match": {"competition_id": identifier}},
        {
            "$project": {
                "club_id": {
                    "$cond": [
                        {"$eq": ["$home_club_id", identifier]},
                        "$home_club_id",
                        "$away_club_id",
                    ]
                },
                "goals_for": {
                    "$cond": [
                        {"$eq": ["$home_club_id", identifier]},
                        "$home_club_goals",
                        "$away_club_goals",
                    ]
                },
                "goals_against": {
                    "$cond": [
                        {"$eq": ["$home_club_id", identifier]},
                        "$away_club_goals",
                        "$home_club_goals",
                    ]
                },
            }
        },
        {
            "$group": {
                "_id": "$club_id",
                "goals_scored": {"$sum": "$goals_for"},
                "goals_conceded": {"$sum": "$goals_against"},
                "goals_difference": {
                    "$sum": {"$subtract": ["$goals_for", "$goals_against"]}
                },
            }
        },
        {"$sort": {"goals_difference": -1}},
    ]

    # Execute the pipeline
    goals_data = list(db.games.aggregate(goals_pipeline))
    goals_df = pd.DataFrame(goals_data)

    # Get club names from PostgreSQL to map club IDs to names
    club_names_query = "SELECT club_id, name FROM clubs"
    club_names_df = pd.read_sql(club_names_query, engine).set_index("club_id")

    # Map the club IDs to names for the goals DataFrame
    goals_df["_id"] = goals_df["_id"].map(club_names_df["name"])

    # Visualization for Goals Scored and Conceded
    plt.figure(figsize=(17, 10))
    sns.scatterplot(
        x="goals_scored", y="goals_conceded", hue="_id", data=goals_df, palette="bright", s=100
    )
    plt.title("Goals Scored vs Conceded")
    plt.xlabel("Goals Scored")
    plt.ylabel("Goals Conceded")

    # Adjust the legend
    leg = plt.legend(
        bbox_to_anchor=(1.01, 1), loc='upper left', labelspacing=1.2, prop={'size': 8}
    )
    # Save the figure
    plt.savefig(os.path.join(output_dir, f"goals_scored_vs_conceded_{identifier}.svg"), bbox_extra_artists=(leg,), bbox_inches='tight')
    
    
    # Experimental hexbin plot
    plt.figure(figsize=(17, 10))
    hb = plt.hexbin(
        goals_df['goals_scored'], 
        goals_df['goals_conceded'], 
        gridsize=30,  # Adjust grid size if necessary
        cmap='YlGnBu',  # A colormap with better contrast
        bins='log',  # Use logarithmic bins for scaling
        mincnt=1,  # Display bins with at least 1 count
        edgecolors='gray',  # Define edges to improve readability
        linewidths=0.5  # Slight line width on edges to improve bin visibility
    )
    cb = plt.colorbar(hb)
    cb.set_label('log10(count in bin)')
    plt.title("Hexbin plot of Goals Scored vs Conceded")
    plt.xlabel("Goals Scored")
    plt.ylabel("Goals Conceded")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"goals_scored_vs_conceded_hexbin_{identifier}.svg"))
    tag_as_advanced_statistic(os.path.join(output_dir, f"goals_scored_vs_conceded_hexbin_{identifier}.svg"))


    attendance_pipeline = [
        {"$match": {"competition_id": identifier}},
        {"$group": {"_id": "$season", "average_attendance": {"$avg": "$attendance"}}},
        {"$sort": {"_id": 1}},  # Sort by season
    ]

    # Execute the pipeline
    attendance_data = list(db.games.aggregate(attendance_pipeline))
    attendance_df = pd.DataFrame(attendance_data)

    # Replace '_id' with 'season' for clarity in the DataFrame
    attendance_df.rename(columns={"_id": "season"}, inplace=True)

    # Visualization for Average Attendance by Season
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x="season",
        y="average_attendance",
        hue="season",
        data=attendance_df,
        palette="muted",
        legend=False,
    )
    plt.title(f"Average Attendance Per Season")
    plt.xlabel("Season")
    plt.ylabel("Average Attendance")
    plt.xticks(rotation=45)
    plt.tight_layout()  # Adjust layout to ensure everything fits
    plt.savefig(os.path.join(output_dir, f"average_attendance_{identifier}.svg"))

    # MongoDB pipeline for Competition Team Standings
    standings_pipeline = [
        {"$match": {"competition_id": identifier}},
        {
            "$project": {
                "home_points": {
                    "$cond": [
                        {"$eq": ["$outcome", "Home Win"]},
                        3,
                        {"$cond": [{"$eq": ["$outcome", "Draw"]}, 1, 0]},
                    ]
                },
                "away_points": {
                    "$cond": [
                        {"$eq": ["$outcome", "Away Win"]},
                        3,
                        {"$cond": [{"$eq": ["$outcome", "Draw"]}, 1, 0]},
                    ]
                },
                "home_club_id": 1,
                "away_club_id": 1,
            }
        },
        {
            "$group": {
                "_id": {
                    "$cond": [
                        {"$gt": ["$home_points", "$away_points"]},
                        "$home_club_id",
                        "$away_club_id",
                    ]
                },
                "points": {
                    "$sum": {
                        "$cond": [
                            {"$gt": ["$home_points", "$away_points"]},
                            "$home_points",
                            "$away_points",
                        ]
                    }
                },
            }
        },
        {"$sort": {"points": -1}},
    ]

    # Execute the pipeline
    standings_data = list(db.games.aggregate(standings_pipeline))
    standings_df = pd.DataFrame(standings_data)

    # Replace '_id' with club names using a lookup from the clubs collection
    club_names_query = "SELECT club_id, name FROM clubs"
    club_names_df = pd.read_sql(club_names_query, engine).set_index("club_id")
    standings_df["_id"] = standings_df["_id"].map(club_names_df["name"])

    # Visualization for Competition Team Standings
    plt.figure(figsize=(20, 15))
    standings_df.set_index("_id", inplace=True)
    standings_df["points"].sort_values().plot(
        kind="barh", legend=False
    )  # Sort the points for better visualization
    plt.title(
        f"Competition Team Standings", fontsize=20
    )
    plt.xlabel(
        "Points: 3 W | 1 D | 0 L", fontsize=14
    ) 
    plt.ylabel("Club", fontsize=14)
    plt.gca().invert_yaxis()  
    plt.xticks(fontsize=12)  
    plt.yticks(fontsize=12)  
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"team_standings_{identifier}.svg"))

    # Advanced Statistics

    # Fetch club names and their IDs from PostgreSQL
    club_names_query = "SELECT club_id, name FROM clubs"
    club_names_df = pd.read_sql(club_names_query, engine).set_index("club_id")

    # MongoDB pipeline for Win/Loss Ratio for each team in a single competition
    win_loss_pipeline = [
        {"$match": {"competition_id": identifier}},
        {
            "$project": {
                "home_win": {"$cond": [{"$eq": ["$outcome", "Home Win"]}, 1, 0]},
                "away_win": {"$cond": [{"$eq": ["$outcome", "Away Win"]}, 1, 0]},
                "home_loss": {"$cond": [{"$eq": ["$outcome", "Away Win"]}, 1, 0]},
                "away_loss": {"$cond": [{"$eq": ["$outcome", "Home Win"]}, 1, 0]},
                "home_club_id": 1,
                "away_club_id": 1,
            }
        },
        {
            "$facet": {
                "home": [
                    {
                        "$group": {
                            "_id": "$home_club_id",
                            "wins": {"$sum": "$home_win"},
                            "losses": {"$sum": "$home_loss"},
                        }
                    }
                ],
                "away": [
                    {
                        "$group": {
                            "_id": "$away_club_id",
                            "wins": {"$sum": "$away_win"},
                            "losses": {"$sum": "$away_loss"},
                        }
                    }
                ],
            }
        },
        {"$project": {"data": {"$concatArrays": ["$home", "$away"]}}},
        {"$unwind": "$data"},
        {"$replaceRoot": {"newRoot": "$data"}},
        {
            "$group": {
                "_id": "$_id",
                "total_wins": {"$sum": "$wins"},
                "total_losses": {"$sum": "$losses"},
            }
        },
        {
            "$project": {
                "_id": 1,
                "total_wins": 1,
                "total_losses": 1,
                "win_loss_ratio": {
                    "$cond": [
                        {"$eq": ["$total_losses", 0]},
                        "Infinity",
                        {"$divide": ["$total_wins", "$total_losses"]},
                    ]
                },
            }
        },
        {"$sort": {"win_loss_ratio": -1}},
    ]

    # Execute the pipeline
    win_loss_data = list(db.games.aggregate(win_loss_pipeline))
    win_loss_df = pd.DataFrame(win_loss_data)

    # Replace MongoDB club IDs with club names from PostgreSQL
    win_loss_df["_id"] = win_loss_df["_id"].map(club_names_df["name"])

    # Calculate the Win/Loss Ratio for each club
    # Avoid division by zero by adding a small number to losses
    win_loss_df["win_loss_ratio"] = win_loss_df["total_wins"] / (
        win_loss_df["total_losses"] + 0.1
    )

    # Sort the DataFrame by 'win_loss_ratio' in descending order for the plot
    sorted_df = win_loss_df.sort_values(by="win_loss_ratio", ascending=False)

    # Bar plot: heights of bars represent Win/Loss Ratio
    plt.figure(figsize=(14, 10))
    plt.bar(
        sorted_df["_id"],
        sorted_df["win_loss_ratio"],
        alpha=0.6,
        color="green",
        edgecolor="black",
    )

    # Adding labels for each club
    for idx, row in sorted_df.iterrows():
        plt.annotate(
            f"{row['win_loss_ratio']:.2f}",
            (row["_id"], row["win_loss_ratio"]),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=9,
        )

    plt.title(f"Win/Loss Ratio by Club", fontsize=16)
    plt.xlabel("Club", fontsize=12)
    plt.ylabel("Win/Loss Ratio", fontsize=12)
    plt.xticks(rotation=90)
    plt.tight_layout()
    filename = f"win_loss_ratio_{identifier}.svg"
    plt.savefig(os.path.join(output_dir, filename))
    tag_as_advanced_statistic(os.path.join(output_dir, filename))

# Games


## Single Game


In [ ]:
# Cell: Single Game Analysis

if stats_category == "game" and identifier:
    game_id = identifier

    # Ensure the output directory exists
    output_dir = f"./outputs/game/{game_id}"
    os.makedirs(output_dir, exist_ok=True)

    # Load game events data from MongoDB
    game_events = list(db.game_events.find({"game_id": identifier}))

    # Load game data from MongoDB
    games_collection = db.games
    game_info = games_collection.find_one({"game_id": identifier})

    if game_info:
        
        # Query to retrieve stadium capacity for the home club
        stadium_capacity_query = f"""
        SELECT stadium_seats FROM clubs
        WHERE club_id = {game_info["home_club_id"]}
        """
        stadium_capacity_result = pd.read_sql(stadium_capacity_query, engine)
        if not stadium_capacity_result.empty:
            stadium_capacity = stadium_capacity_result.iloc[0]['stadium_seats']
        else:
            print(f"No stadium capacity found for club ID {game_info['home_club_id']}")
            stadium_capacity = 0  # Default to 0 if not found

        # Preparing data for visualization
        teams = [game_info["home_club_name"], game_info["away_club_name"]]
        goals = [game_info["home_club_goals"], game_info["away_club_goals"]]
        
        # Create the timeline visualization
        fig, ax = plt.subplots(figsize=(10, 2))

        # Dynamically adjust the limits of the x-axis based on the events
        extra_time = 5  # Additional minutes for padding
        max_minute = max(event.get("minute", 0) for event in game_events) + extra_time
        ax.set_xlim(0, max_minute)
        ax.set_ylim(-2, 2)
        ax.axhline(0, color='grey', lw=2)

        # Define a function to categorize events
        def categorize_event(event):
            event_type = event['type'].lower()
            description = event['description'].lower()
            if 'yellow card' in description:
                return 'yellow_card'
            elif 'red card' in description:
                return 'red_card'
            else:
                return event_type.replace(" ", "_")

        # Plot each event on the timeline
        for event in game_events:
            y = -1 if event["club_id"] == game_info["home_club_id"] else 1
            category = categorize_event(event)
            color = {
                "goals": "green",
                "yellow_card": "#d4ca1c",
                "red_card": "red",
                "substitutions": "blue",
                "shootout": "black"
            }.get(category, "grey")
            marker = {
                "goals": '^',
                "yellow_card": 'o',
                "red_card": 'o',
                "substitutions": 's',
                "shootout": 'x'
            }.get(category, 'x')
            minute = event.get("minute", 0)

            ax.scatter(minute, y, s=100, color=color, marker=marker, label=category.capitalize())

        # Customizations
        ax.set_xlabel("Minute")
        ax.set_yticks([])
        ax.set_xticks(np.arange(0, max_minute + 1, 10))
        ax.set_title(f"Timeline of Game Events")
        plt.tight_layout()

        # Creating a custom legend manually
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='^', color='w', label='Goal', markerfacecolor='green', markersize=10),
            Line2D([0], [0], marker='o', color='w', label='Yellow Card', markerfacecolor='#d4ca1c', markersize=10),
            Line2D([0], [0], marker='o', color='w', label='Red Card', markerfacecolor='red', markersize=10),
            Line2D([0], [0], marker='s', color='w', label='Substitution', markerfacecolor='blue', markersize=10),
        ]

        # Place the legend inside the plot for better SVG rendering
        ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.15),
                fancybox=True, shadow=True, ncol=4)

        # Save the figure ensuring all content is included in the SVG
        plt.savefig(os.path.join(output_dir, f"game_events_timeline_{identifier}.svg"), format='svg', bbox_inches='tight')

        # Attendance visualization
        attendance = game_info.get("attendance", 0)  # Get attendance or default to 0
        if attendance >= 0 and stadium_capacity > 0:
            # Create the pie chart for attendance
            plt.figure(figsize=(6, 6))
            plt.pie(
                [attendance, stadium_capacity - attendance],
                labels=["Attendance", "Remaining Capacity"],
                autopct='%1.1f%%',
                startangle=140,
                colors=["green", "grey"]
            )
            plt.title(f"Stadium Attendance")
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"attendance_{identifier}.svg"))
        else:
            print(f"Invalid attendance data or stadium capacity for game ID {identifier}")
            
            
        # Advanced Statistics
        home_club_id = game_info["home_club_id"]
        away_club_id = game_info["away_club_id"]

        # Fetch historical matches
        historical_matches = list(db.games.find({
            "$or": [
                {"$and": [{"home_club_id": home_club_id}, {"away_club_id": away_club_id}]},
                {"$and": [{"home_club_id": away_club_id}, {"away_club_id": home_club_id}]}
            ]
        }))

        # Initialize counters
        home_wins = away_wins = draws = 0

        # Count outcomes
        for match in historical_matches:
            if match["outcome"] == "Home Win":
                if match["home_club_id"] == home_club_id:
                    home_wins += 1
                else:
                    away_wins += 1
            elif match["outcome"] == "Away Win":
                if match["away_club_id"] == away_club_id:
                    away_wins += 1
                else:
                    home_wins += 1
            elif match["outcome"] == "Draw":
                draws += 1

        # Calculate total games and winning percentage for the home club
        total_games = len(historical_matches)
        if total_games > 0:
            winning_percentage = ((home_wins + (draws * 0.5)) / total_games) * 100
            winning_percentage = round(winning_percentage, 2)
        else:
            winning_percentage = None
            
        # Create a figure for the gauge chart
        fig, ax = plt.subplots(figsize=(4, 2), subplot_kw={'aspect': 'auto'})

        # Create the pie chart as the base of the gauge
        sizes = [winning_percentage, 100 - winning_percentage]
        colors = ['#0f4c81', '#e5e5e5']  # Use more appealing colors
        start_angle = 90
        ax.pie(sizes, colors=colors, startangle=start_angle, radius=0.9,
            wedgeprops=dict(width=0.3, edgecolor='w'))

        # Add a circle at the center to create a donut hole
        from matplotlib.patches import Circle
        circle = Circle((0, 0), 0.7, color='white', ec='white')
        ax.add_artist(circle)

        # Calculate the angle for the winning percentage
        angle = (360 * winning_percentage) / 100
        # Draw a bar to represent the winning percentage
        ax.barh(0, 0.85, left=start_angle, height=0.3, color='#0f4c81',
                align='center', edgecolor='white')

        # Remove y-axis and spines
        ax.set_yticks([])
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Set x-axis limits to make the gauge look half-circle
        ax.set_xlim(-0.5, 0.5)

        # Add the percentage as text in the middle of the donut hole
        plt.text(0, 0, f'{winning_percentage}%', ha='center', va='center', fontsize=12)

        # Add a title
        plt.title('Home Club Winning Percentage\nBased on Previous Encounters', pad=20)

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"winning_percentage_{identifier}.svg"))
        tag_as_advanced_statistic(os.path.join(output_dir, f"winning_percentage_{identifier}.svg"))
else:
    print(f"No data found for game ID {identifier}")
